In [8]:
import torch
print("✅ PyTorch ready!")
print("PyTorch version:", torch.__version__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


✅ PyTorch ready!
PyTorch version: 2.5.1+cpu
Device: CPU


In [9]:
import os
from pathlib import Path
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [ ]:
from pathlib import Path


data_dir = Path("PlantDiseasesDataset")
train_dir = data_dir / "Train"
val_dir = data_dir / "Validation"


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)

print("🎉 DATASET LOADED!")
print(f"Train images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Classes: {num_classes}")
print("Sample classes:", class_names[:5])


🎉 DATASET LOADED!
Train images: 1940
Validation images: 160
Classes: 8
Sample classes: ['Bacterial_Spot', 'Black_Measles', 'Black_Rot', 'Gray_Leaf_Spot', 'Healthy']


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models


model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


model.fc = nn.Linear(model.fc.in_features, num_classes)


model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

save_path = "plant_disease_model.pth"

print(f"✅ MODEL READY!")
print(f"Classes: {num_classes}")
print(f"Model on: {device}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\pmuke/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


✅ MODEL READY!
Classes: 8
Model on: cpu


In [ ]:

save_path = "plant_disease_model.pth"
num_epochs = 3 
batch_size = 16  


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

best_val_acc = 0
print("⚡ FAST TRAINING MODE")
print(f"Batch size: {batch_size}, Epochs: {num_epochs}")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    
    model.train()
    train_correct = 0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        
        if batch_idx % 20 == 0:  
            print(f"  Batch {batch_idx}/{len(train_loader)}")
    
    train_acc = train_correct / len(train_dataset)
    
    
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
    
    val_acc = val_correct / len(val_dataset)
    print(f"  Train: {train_acc:.3f} | Val: {val_acc:.3f}")
    
   
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': class_names,
        }, save_path)
        print(f"  💾 SAVED: {val_acc:.3f}")

print(f"\n🎉 FAST TRAINING DONE! Best: {best_val_acc:.3f}")
print("Model ready for prediction!")


⚡ FAST TRAINING MODE
Batch size: 16, Epochs: 3

Epoch 1/3
  Batch 0/122
  Batch 20/122
  Batch 40/122
  Batch 60/122
  Batch 80/122
  Batch 100/122
  Batch 120/122
  Train: 0.815 | Val: 0.775
  💾 SAVED: 0.775

Epoch 2/3
  Batch 0/122
  Batch 20/122
  Batch 40/122
  Batch 60/122
  Batch 80/122
  Batch 100/122
  Batch 120/122
  Train: 0.875 | Val: 0.806
  💾 SAVED: 0.806

Epoch 3/3
  Batch 0/122
  Batch 20/122
  Batch 40/122
  Batch 60/122
  Batch 80/122
  Batch 100/122
  Batch 120/122
  Train: 0.899 | Val: 0.856
  💾 SAVED: 0.856

🎉 FAST TRAINING DONE! Best: 0.856
Model ready for prediction!
